# Documentacion de Archivos — Dofbot ROS 2
> **Workspace:** ws_ts26_2 | **ROS 2 Jazzy** | **NVIDIA Jetson**
---


---
## 1. dofbot_interfaces — Tipos de datos personalizados
> **Por que existe:** ROS 2 necesita saber que es una "telemetria" o un "gripper command".
> Este paquete define esos tipos con archivos .msg, .srv y .action.


### 1.1 msg/Telemetry.msg — Mensaje de posicion y estado

In [ ]:
# Contenido del archivo Telemetry.msg
telemetry_msg = """
string status      # Estado textual: "Active", "STAND BY"
float32 pos_x      # Posicion X del efector final (metros)
float32 pos_y
float32 pos_z
"""
print(telemetry_msg)

# Uso en Python:
# from dofbot_interfaces.msg import Telemetry
# msg = Telemetry()
# msg.status = "Active"
# msg.pos_x = 0.15


### 1.2 srv/GetStatus.srv — Servicio de estado del robot

In [ ]:
# Estructura del servicio: separada por ---
get_status_srv = """
# REQUEST (lo que el cliente envia)
bool is_robot_active
---
# RESPONSE (lo que el servidor devuelve)
bool is_active
bool success
string string_status_message
"""
print(get_status_srv)


### 1.3 action/GripperCmd.action — Accion para mover el gripper

In [ ]:
gripper_action = """
# GOAL (meta que el cliente pide)
float32 OPEN  = -1.4209   # constante: gripper abierto
float32 CLOSE =  0.0      # constante: gripper cerrado
float32 gripper_state     # valor objetivo
float32 duration          # segundos para completarlo
---
# RESULT (al terminar la tarea)
float32 current_state
bool success
string string_status_message
---
# FEEDBACK (progreso en tiempo real)
float32 current_state
"""
print(gripper_action)

# Diferencia clave con un servicio:
# Servicio  = peticion rapida, respuesta inmediata
# Accion    = tarea larga, con progreso intermedio y cancelacion posible


### 1.4 CMakeLists.txt — Generacion de codigo desde los archivos de interfaz

In [ ]:
cmake_snippet = """
rosidl_generate_interfaces(${PROJECT_NAME}
  "msg/Telemetry.msg"
  "srv/GetStatus.srv"
  "action/GripperCmd.action"
)
# colcon genera automaticamente los modulos Python/C++:
#   dofbot_interfaces.msg.Telemetry
#   dofbot_interfaces.srv.GetStatus
#   dofbot_interfaces.action.GripperCmd
"""
print(cmake_snippet)


---
## 2. dofbot_config — Servidor de parametros
> **Concepto clave:** Los parametros en ROS 2 son variables de configuracion que viven
> dentro de un nodo. Pueden leerse y modificarse desde fuera sin recompilar.


### 2.1 parameter_server.py — Nodo que gestiona la configuracion del robot

In [ ]:
parameter_server_explained = {
    "clase": "DofbotParamSrv(Node)",
    "patron": "Hereda de rclpy.node.Node",
    "pasos": [
        "1. declare_parameter()  registra 'time_period' con valor por defecto 0.01",
        "2. declare_parameters() registra: vel_lin, vel_ang, joint_names, robot_ip, robot_name",
        "3. get_parameter()      lee el valor actual de un parametro",
        "4. add_on_set_parameters_callback()  activa validacion ANTES de aplicar cambios",
    ],
    "validaciones": {
        "time_period": "debe ser >= 0.0",
        "robot_ip": "debe ser IPv4 valida (regex)"
    }
}

for k, v in parameter_server_explained.items():
    print(f"  {k}: {v}")


In [ ]:
import re

IPV4_REGEX = r"^((25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])\.){3}(25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])$"

def validate_ip(ip):
    return bool(re.match(IPV4_REGEX, ip))

test_ips = ["192.168.200.128", "300.1.1.1", "10.0.0.1", "abc.def.ghi.jkl"]
for ip in test_ips:
    estado = "valida" if validate_ip(ip) else "invalida"
    print(f"  {ip:20s} = {estado}")


### 2.2 launch/param_srv.launch.py — Lee variables de entorno y lanza el nodo

In [ ]:
launch_explained = """
import os

# Lee del entorno: si no existe, usa valor por defecto
robot_name = os.getenv('ROBOT_NAME', 'VIRTUAL')
robot_ip   = os.getenv('IPADDR', '127.0.0.1')

# namespace = robot_name en minusculas
# Todos los topicos quedan bajo /dofbot_arm/...
param_srv_node = Node(
    package   = "dofbot_config",
    executable = "param_srv",
    namespace  = robot_name.lower(),
    parameters = [{"robot_name": robot_name, "robot_ip": robot_ip}]
)
"""
print(launch_explained)


### 2.3 config/dofbot_params.yaml — Valores por defecto

In [ ]:
yaml_content = """
/dofbot_config:
  ros__parameters:
    joint_names: [base_joint, arm_joint01, arm_joint_02,
                  arm_joint_03, arm_joint_04, arm_joint_05, gripper_joint]
    robot_ip: 192.168.200.128
    robot_name: VIRTUAL
    vel_ang: 0.01
    vel_lin: 0.01
"""
print("Estructura del YAML de parametros:")
print(yaml_content)
print("Nota: la clave /dofbot_config = namespace del nodo")
print("      ros__parameters es la convencion de ROS 2 para YAML")


---
## 3. dofbot_control — Action Server del Gripper
> **Concepto clave:** Action = servicio con feedback.
> Util para movimientos que tardan varios segundos y necesitan reportar progreso.


### 3.1 DofbotSimpleActionServer.py — Ciclo completo Goal-Execute-Result

In [ ]:
action_flow = {
    "1_recibir_goal"  : "goal_handle.request contiene gripper_state y duration",
    "2_validar_rango" : [
        "validate_range(gripper_state, OPEN=-1.4209, CLOSE=0.0)",
        "validate_range(duration, min=0.0, max=10.0)",
        "Si falla: goal_handle.abort() y retorna Result con success=False"
    ],
    "3_ejecutar"      : [
        "Calcula delta = (goal_state - estado_actual) / duration",
        "Loop hasta agotar el tiempo:",
        "  publica GripperCmd.Feedback con estado intermedio",
        "  espera 1 segundo",
    ],
    "4_resultado"     : "goal_handle.succeed() + Result(success=True)"
}

for step, detail in action_flow.items():
    print(f"
  [{step}]")
    if isinstance(detail, list):
        for d in detail: print(f"    - {d}")
    else:
        print(f"    {detail}")


In [ ]:
# Como llamar al Action Server desde un cliente:
client_code = """
from rclpy.action import ActionClient
from dofbot_interfaces.action import GripperCmd

client = ActionClient(node, GripperCmd, "gripper_command")

goal = GripperCmd.Goal()
goal.gripper_state = GripperCmd.Goal.OPEN   # -1.4209
goal.duration = 3.0                          # 3 segundos

# Envia la meta de forma asincrona
future = client.send_goal_async(
    goal,
    feedback_callback=lambda fb: print(fb.feedback.current_state)
)
"""
print(client_code)


---
## 4. dofbot_services — Servicios ROS 2
> **Concepto clave:** Comunicacion sincrona request-response.
> El cliente bloquea hasta recibir respuesta o agotar el timeout.


### 4.1 dofbot_server.py — Nodo servidor

In [ ]:
server_pattern = """
class DofbotServServer(Node):

    def __init__(self):
        # Registra el servicio con su callback
        self.create_service(
            srv_type = GetStatus,
            srv_name = "/dofbot_status_srv",
            callback = self._on_status_srv_clbk
        )

    def _on_status_srv_clbk(self, request, response):
        # request.is_robot_active  = bool enviado por el cliente
        response.is_active = request.is_robot_active
        response.success = True
        response.string_status_message = "Todo ok"
        return response   # SIEMPRE retornar el response
"""
print(server_pattern)


### 4.2 dofbot_client.py — Nodo cliente con reintentos

In [ ]:
client_pattern = """
def call_service(self, is_active):
    # Espera hasta 3 veces que el servidor aparezca
    while not self.__status_client.wait_for_service(timeout_sec=1.0):
        if self.__wait_count == 0:
            return None          # agoto reintentos
        self.__wait_count -= 1

    peticion = GetStatus.Request()
    peticion.is_robot_active = is_active

    # Llama de forma asincrona y espera el resultado
    self.future = self.__status_client.call_async(peticion)
    rclpy.spin_until_future_complete(self, self.future)
    return self.future.result()
"""
print(client_pattern)


---
## 5. dofbot_telemetry — Telemetria del sistema
> Convierte informacion del SO (CPU, RAM, disco) en mensajes DiagnosticArray,
> el estandar de ROS 2 para reportar estado de hardware.


### 5.1 Telemetry.py — Publicador simple

In [ ]:
pub_pattern = """
class TelemetryNode(Node):
    def __init__(self):
        # Publicador en /telemetry, cola de 10 mensajes
        self._telemetry_pub = self.create_publisher(Telemetry, "/telemetry", 10)
        # Timer: ejecuta el callback cada 1.0 segundos
        self._telemetry_timer = self.create_timer(1.0, self._on_telemetry_clbk)

    def _on_telemetry_clbk(self):
        msg = Telemetry()
        msg.status = "Active"
        msg.pos_x = msg.pos_y = msg.pos_z = 0.0
        self._telemetry_pub.publish(msg)
"""
print(pub_pattern)


### 5.2 TelemetrySubs.py — Suscriptor simple

In [ ]:
sub_pattern = """
class TelemetrySubsNode(Node):
    def __init__(self):
        # Suscriptor en /telemetry, cola de 10 mensajes
        self._telem_sub = self.create_subscription(
            Telemetry, "/telemetry", self._on_telem_clbk, 10
        )

    def _on_telem_clbk(self, telem_msg):
        # Se ejecuta CADA VEZ que llega un mensaje
        self.get_logger().info(
            f"Recibi: [{telem_msg.status}] pos: [{telem_msg.pos_x}, ...]")
"""
print(sub_pattern)


### 5.3 robot_telem.py — Telemetria completa con DiagnosticArray

In [ ]:
robot_telem_flow = [
    "1. SysInfo().get_system_report()  obtiene todos los datos del SO",
    "2. _find_key_recursive()  extrae cpu_stats, disk, ram, ip, ros del reporte",
    "3. Por cada nucleo CPU  crea DiagnosticStatus con % de uso",
    "4. Para disco y RAM  crea DiagnosticStatus con size/used/available",
    "5. Para IP  crea DiagnosticStatus con la direccion actual",
    "6. Para ROS  agrega version, distro, domain_id y nombre del robot",
    "7. DiagnosticArray.status = todos los anteriores",
    "8. __diag_pub.publish(array)  publica en /diagnostics"
]

print("Flujo de robot_telem.py:")
for step in robot_telem_flow:
    print(f"  {step}")


In [ ]:
# Funcion clave: busqueda recursiva en dicts anidados
def _find_key_recursive(data, target_key):
    if isinstance(data, dict):
        if target_key in data:
            return data[target_key]
        for key, value in data.items():
            result = _find_key_recursive(value, target_key)
            if result is not None:
                return result
    elif isinstance(data, list):
        for item in data:
            result = _find_key_recursive(item, target_key)
            if result is not None:
                return result
    return None

# Prueba con un reporte simulado
reporte = {
    "host": "jetson",
    "hardware": {
        "disk": {"size": 59, "used": 14, "available": 43},
        "ram":  {"total": 3.9, "used": 1.8}
    }
}

print("disk:", _find_key_recursive(reporte, "disk"))
print("ram: ", _find_key_recursive(reporte, "ram"))
print("cpu: ", _find_key_recursive(reporte, "cpu"))   # No existe -> None


### 5.4 jtop_telem.py — Telemetria Jetson con jtop

In [ ]:
jtop_telem_summary = {
    "libreria": "jtop (pip install jetson-stats)",
    "que publica": [
        "CPU por nucleo (freq, governor, idle)",
        "GPU (load, freq)",
        "RAM / SWAP / EMC",
        "Temperatura por sensor",
        "Potencia (rail y total en mW)",
        "Velocidad del fan",
        "Info del board (Jetpack, L4T)",
        "Estado del disco"
    ],
    "topic_salida": "nv_diagnostics (DiagnosticArray)",
    "nota": "Requiere hardware fisico Jetson para funcionar"
}

for k, v in jtop_telem_summary.items():
    print(f"
  {k}:")
    if isinstance(v, list):
        for item in v: print(f"    - {item}")
    else:
        print(f"    {v}")


---
## 6. arrg_utils/sysinfo.py — Libreria de informacion del sistema
> Independiente de ROS. Usa subprocess para ejecutar comandos de shell
> y leer /proc/stat, free, df, ip.


In [ ]:
sysinfo_api = {
    "get_host_info()": "hostname + IP principal",
    "get_free_disk()": "size/used/available en GB (desde df)",
    "get_free_ram()": "total/used/free/available en GB (desde free -h)",
    "get_system_date()": "date y time actuales",
    "get_cpu_usage()": "lista de stats por nucleo (desde /proc/stat)",
    "get_ros_info()": "ROS_VERSION, ROS_DISTRO, ROS_DOMAIN_ID",
    "get_system_report()": "dict completo con todo lo anterior",
    "get_system_snapshot()": "dict resumido: cpu%, ram, disk, ip, time"
}

print("API de SysInfo:")
for method, desc in sysinfo_api.items():
    print(f"  {method:35s} {desc}")


In [ ]:
# Como ejecuta comandos shell internamente
import subprocess

def execute_command(command):
    try:
        result = subprocess.check_output(command, shell=True)
        return result.decode().strip()
    except subprocess.CalledProcessError as e:
        print(f"Error: {e}")
        return None

# Calculo de uso de CPU
def compute_cpu_usage(user_val, system_val, idle_val):
    # user + system = trabajo real
    # idle = tiempo sin hacer nada
    # uso% = trabajo / total * 100
    total = user_val + system_val + idle_val
    trabajo = user_val + system_val
    return float(trabajo) * 100 / total

# Ejemplo con valores tipicos de Jetson Nano
user, system, idle = 171549, 57426, 943318
uso = compute_cpu_usage(user, system, idle)
print(f"CPU usage: {uso:.2f}%")


---
## 7. URDF / Xacro — Descripcion del robot
> URDF es el XML que describe la geometria, colision e inercia de cada parte.
> Xacro agrega variables y macros para no repetir codigo.


In [ ]:
xacro_concepts = {
    "xacro:property": "Variable. Ej: wheel_radius = 0.1",
    "xacro:macro":    "Funcion reutilizable. Ej: macro wheel_link(prefix)",
    "xacro:include":  "Incluye otro archivo .xacro",
    "${expresion}":  "Evalua la expresion. Ej: ${wheel_radius * 2}",
}

link_partes = {
    "<visual>":    "Como se ve: geometria + material",
    "<collision>": "Volumen para detectar choques fisicos",
    "<inertial>": "Masa y matriz de inercia (necesaria para simulacion)",
}

joint_tipos = {
    "fixed":      "No se mueve (base unida al piso)",
    "revolute":   "Gira con limites. Ej: brazo",
    "continuous": "Gira sin limites. Ej: ruedas",
    "prismatic":  "Desliza linealmente",
}

print("Conceptos Xacro:")
for k, v in xacro_concepts.items():
    print(f"  {k:20s}: {v}")

print("
Partes de un <link>:")
for k, v in link_partes.items():
    print(f"  {k:15s}: {v}")

print("
Tipos de <joint>:")
for k, v in joint_tipos.items():
    print(f"  {k:15s}: {v}")


In [ ]:
# Macro de inercia para caja — common_properties.xacro
inertia_macro = """
<xacro:macro name="box_inertia" params="m x y z o_xyz o_rpy">
    <inertial>
        <mass value="${m}" />
        <!-- Formulas de inercia para caja solida: -->
        <inertia ixx="${(m/12) * (y*y + z*z)}"
                 iyy="${(m/12) * (x*x + z*z)}"
                 izz="${(m/12) * (x*x + y*y)}"
                 ixy="0" iyz="0" ixz="0" />
    </inertial>
</xacro:macro>

<!-- Uso: -->
<xacro:box_inertia m="5.0"
    x="${base_length}" y="${base_width}" z="${base_height}"
    o_xyz="0 0 ${base_height/2}" o_rpy="0 0 0" />
"""
print(inertia_macro)


---
## 8. dockerimg/ — Contenedor para Jetson
> Docker aisla el entorno ROS 2 del SO base del Jetson.
> Facilita el despliegue y garantiza reproducibilidad.


In [ ]:
dockerfile_stages = {
    "Stage 1 Builder": [
        "Base: dustynv/ros:jazzy-ros-base-r36.4.0 (imagen oficial para Jetson)",
        "Fix de llaves ROS2 (migracion 2024)",
        "Instala: python3-pip, smbus, PIL, i2c-tools, rmw-cyclonedds",
    ],
    "Stage 2 Runtime": [
        "USER ubuntu (no root por seguridad)",
        "COPY bash_aliases: aliases srcthis y ros2path",
        "ENV ROS_DOMAIN_ID=10",
        "ENV RMW_IMPLEMENTATION=rmw_cyclonedds_cpp",
        "CMD bash"
    ]
}

for stage, steps in dockerfile_stages.items():
    print(f"
  [{stage}]")
    for s in steps: print(f"    {s}")

print("
Docker Compose - puntos clave:")
compose_points = {
    "runtime: nvidia":    "Acceso a GPU CUDA dentro del contenedor",
    "network_mode: host": "Comparte red del host, DDS funciona sin config extra",
    "shm_size: 8g":       "Memoria compartida para camara y sensores rapidos",
    "volumes":            "dofbotx -> /home/robot (codigo) + sockets Jetson",
    "devices":            "video0 (camara) + i2c-0~8 (servos) + usb"
}
for k, v in compose_points.items():
    print(f"  {k:25s}: {v}")


---
## 9. dofbot_telemetry_cpp — Comparacion Python vs C++


In [ ]:
cpp_vs_python = {
    "Herencia":         ("class TelemetryPub : public rclcpp::Node",
                          "class TelemetryNode(Node)"),
    "Constructor":      ("TelemetryPub() : Node("telemetry_node") {}",
                          "super().__init__("telemetry_node")"),
    "Publicador":       ("create_publisher<Telemetry>("/tel", 10)",
                          "create_publisher(Telemetry, "/tel", 10)"),
    "Timer":            ("create_wall_timer(1s, std::bind(&Class::cb, this))",
                          "create_timer(1.0, self.cb)"),
    "Log":              ("RCLCPP_INFO(get_logger(), "msg")",
                          "self.get_logger().info("msg")"),
    "Init/Spin":        ("rclcpp::init / rclcpp::spin",
                          "rclpy.init / rclpy.spin"),
}

print(f"{'Concepto':<15} {'C++':<50} {'Python'}")
print("-" * 100)
for concepto, (cpp, py) in cpp_vs_python.items():
    print(f"{concepto:<15} {cpp:<50} {py}")


---
## 10. Comandos utiles para verificar el sistema


In [ ]:
comandos = {
    "Ver topicos activos":        "ros2 topic list",
    "Escuchar telemetria":         "ros2 topic echo /telemetry",
    "Ver diagnosticos":            "ros2 topic echo /diagnostics",
    "Listar servicios":            "ros2 service list",
    "Llamar servicio": (
        "ros2 service call /dofbot_status_srv "
        "dofbot_interfaces/srv/GetStatus '{is_robot_active: true}'"
    ),
    "Ver parametros del nodo":     "ros2 param list /dofbot_arm/dofbot_config",
    "Cambiar un parametro":        "ros2 param set /dofbot_arm/dofbot_config vel_lin 0.05",
    "Ver action servers":          "ros2 action list",
    "Enviar meta al gripper": (
        "ros2 action send_goal /gripper_command "
        "dofbot_interfaces/action/GripperCmd '{gripper_state: -1.4209, duration: 3.0}'"
    ),
}

print("Comandos de diagnostico:")
for desc, cmd in comandos.items():
    print(f"
  # {desc}")
    print(f"  $ {cmd}")
